# MNIST Exploration Notebook

This notebook helps you understand the MNIST dataset and experiment interactively.

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Load and Explore the Data

In [ ]:
# Load MNIST
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root='../data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root='../data',
    train=False,
    download=True,
    transform=transform
)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

## 2. Visualize Random Samples

In [ ]:
# Display random samples
fig, axes = plt.subplots(3, 6, figsize=(12, 6))
axes = axes.ravel()

for i in range(18):
    idx = np.random.randint(0, len(train_dataset))
    image, label = train_dataset[idx]
    
    axes[i].imshow(image.squeeze(), cmap='gray')
    axes[i].set_title(f'Label: {label}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 3. Analyze the Dataset

In [ ]:
# Count samples per class
labels = [label for _, label in train_dataset]
unique, counts = np.unique(labels, return_counts=True)

plt.figure(figsize=(10, 5))
plt.bar(unique, counts)
plt.xlabel('Digit')
plt.ylabel('Count')
plt.title('Distribution of Digits in Training Set')
plt.xticks(unique)
plt.grid(axis='y', alpha=0.3)
plt.show()

print("Samples per class:")
for digit, count in zip(unique, counts):
    print(f"  {digit}: {count}")

## 4. Pixel Value Distribution

In [ ]:
# Analyze pixel values
sample_image, _ = train_dataset[0]
pixels = sample_image.numpy().flatten()

plt.figure(figsize=(10, 5))
plt.hist(pixels, bins=50, edgecolor='black')
plt.xlabel('Pixel Value')
plt.ylabel('Frequency')
plt.title('Pixel Value Distribution')
plt.grid(axis='y', alpha=0.3)
plt.show()

print(f"Min pixel value: {pixels.min()}")
print(f"Max pixel value: {pixels.max()}")
print(f"Mean pixel value: {pixels.mean():.4f}")
print(f"Std pixel value: {pixels.std():.4f}")

## 5. Interactive Model Building

Try modifying this simple model!

In [ ]:
class CustomNN(nn.Module):
    def __init__(self, hidden_size=128):
        super(CustomNN, self).__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 10)
        )
    
    def forward(self, x):
        return self.network(x)

# Create model and count parameters
model = CustomNN(hidden_size=128)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 6. Quick Training Test

Train for just 1 epoch to see if everything works!

In [ ]:
from torch.utils.data import DataLoader
import torch.optim as optim

# Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CustomNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Train one epoch
model.train()
for batch_idx, (images, labels) in enumerate(train_loader):
    images, labels = images.to(device), labels.to(device)
    
    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    
    if batch_idx % 100 == 0:
        print(f"Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}")

print("\nTraining test complete!")

## Experiments to Try

1. Change `hidden_size` in CustomNN to see how it affects parameter count
2. Add more layers to the network
3. Try different activation functions (Tanh, LeakyReLU, etc.)
4. Visualize what the model learns after training
5. Implement data augmentation (rotations, scaling)

Have fun exploring!